# Analyst Report

This notebook builds an analyst report for the security SaaS using the Postgres `events` and `alerts` tables.


## Setup


In [ ]:
import os
import json
import math
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sqlalchemy import create_engine, text

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

import warnings
warnings.filterwarnings("ignore")

plt.style.use("ggplot")


In [ ]:
DATABASE_URL = os.getenv("DATABASE_URL")
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASS = os.getenv("DB_PASS", "postgres")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "shield_db")


def get_engine():
    url = DATABASE_URL or f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    return create_engine(url)


## Data Access


In [ ]:
engine = get_engine()

try:
    with engine.connect() as conn:
        events = pd.read_sql(
            text('SELECT ts, ts_epoch, source, event_type, "user", host, ip, raw FROM events'),
            conn,
        )
        alerts = pd.read_sql(
            text('SELECT rule_id, severity, "group", evidence FROM alerts'),
            conn,
        )
except Exception as exc:
    raise RuntimeError(f"Failed to read Postgres tables: {exc}")

print(f"Loaded events: {len(events):,}")
print(f"Loaded alerts: {len(alerts):,}")


In [ ]:
def normalize_json(value):
    if isinstance(value, dict):
        return value
    if value is None:
        return {}
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return {}
    return {}


events["raw_json"] = events["raw"].apply(normalize_json)
alerts["evidence_json"] = alerts["evidence"].apply(normalize_json)


def extract_last_event(evidence):
    if not isinstance(evidence, dict):
        return {}
    last_event = evidence.get("last_event") or evidence.get("event") or {}
    if isinstance(last_event, str):
        last_event = normalize_json(last_event)
    if not isinstance(last_event, dict):
        return {}
    return last_event


alerts["last_event"] = alerts["evidence_json"].apply(extract_last_event)
alerts["alert_ip"] = alerts["last_event"].apply(
    lambda x: x.get("ip") or x.get("src_ip") or x.get("source_ip")
)
alerts["alert_ts_epoch"] = alerts["last_event"].apply(
    lambda x: x.get("ts_epoch") or x.get("timestamp_epoch")
)
alerts["alert_ts"] = alerts["last_event"].apply(lambda x: x.get("ts"))
alerts["alert_ts_dt"] = pd.to_datetime(alerts["alert_ts_epoch"], unit="s", errors="coerce")
alerts.loc[alerts["alert_ts_dt"].isna(), "alert_ts_dt"] = pd.to_datetime(
    alerts.loc[alerts["alert_ts_dt"].isna(), "alert_ts"], errors="coerce"
)


def extract_geo(evidence):
    if not isinstance(evidence, dict):
        return {}
    enrich = evidence.get("enrich") or {}
    geo = enrich.get("geoip") or evidence.get("geoip") or {}
    if isinstance(geo, str):
        geo = normalize_json(geo)
    if not isinstance(geo, dict):
        return {}
    return geo


alerts["geo"] = alerts["evidence_json"].apply(extract_geo)
alerts["geo_country"] = alerts["geo"].apply(
    lambda x: x.get("country") or x.get("country_iso") or x.get("country_code")
)
alerts["geo_asn"] = alerts["geo"].apply(lambda x: x.get("asn") or x.get("as_number"))
alerts["geo_lat"] = alerts["geo"].apply(lambda x: x.get("latitude") or x.get("lat"))
alerts["geo_lon"] = alerts["geo"].apply(
    lambda x: x.get("longitude") or x.get("lon") or x.get("lng")
)

geo_lookup = (
    alerts.dropna(subset=["alert_ip"]).groupby("alert_ip", as_index=False).agg(
        {
            "geo_country": "first",
            "geo_asn": "first",
            "geo_lat": "first",
            "geo_lon": "first",
        }
    )
)

geo_lookup = geo_lookup.rename(columns={"alert_ip": "ip"})

events["ts_dt"] = pd.to_datetime(events["ts_epoch"], unit="s", errors="coerce")
if events["ts_dt"].isna().all() and "ts" in events.columns:
    events["ts_dt"] = pd.to_datetime(events["ts"], errors="coerce")

events = events.merge(geo_lookup, on="ip", how="left")


## Executive Summary


In [ ]:
from IPython.display import Markdown, display


def fmt_ts(series):
    series = series.dropna()
    if series.empty:
        return "n/a"
    return f"{series.min()} to {series.max()}"


event_range = fmt_ts(events["ts_dt"])
alert_range = fmt_ts(alerts["alert_ts_dt"])

summary_lines = []
summary_lines.append(f"- Total events: {len(events):,}")
summary_lines.append(f"- Total alerts: {len(alerts):,}")
summary_lines.append(f"- Event time range: {event_range}")
summary_lines.append(f"- Alert time range: {alert_range}")

if not alerts.empty:
    top_rules = alerts["rule_id"].value_counts().head(5)
    summary_lines.append("- Top rule_id: " + ", ".join([f"{idx} ({val})" for idx, val in top_rules.items()]))
else:
    summary_lines.append("- Top rule_id: n/a")

for label, col in [("IPs", "ip"), ("Users", "user"), ("Hosts", "host")]:
    if col in events.columns:
        top_vals = events[col].value_counts().head(5)
        summary_lines.append(
            f"- Top {label}: " + ", ".join([f"{idx} ({val})" for idx, val in top_vals.items()])
        )

if not alerts.empty:
    top_countries = alerts["geo_country"].value_counts().head(5)
    if not top_countries.empty:
        summary_lines.append(
            "- Top countries: " + ", ".join([f"{idx} ({val})" for idx, val in top_countries.items()])
        )
    else:
        summary_lines.append("- Top countries: n/a")

    top_asns = alerts["geo_asn"].value_counts().head(5)
    if not top_asns.empty:
        summary_lines.append(
            "- Top ASNs: " + ", ".join([f"{idx} ({val})" for idx, val in top_asns.items()])
        )
    else:
        summary_lines.append("- Top ASNs: n/a")

summary_md = "
".join(summary_lines)

display(Markdown(summary_md))


## Data Quality Checks


In [ ]:
quality = {}

required_event_fields = ["ts", "ts_epoch", "source", "event_type", "user", "host", "ip"]
required_alert_fields = ["rule_id", "severity", "group", "evidence"]

quality["events_missing"] = (
    events[required_event_fields].isna().sum().sort_values(ascending=False)
    if not events.empty
    else pd.Series(dtype=int)
)
quality["alerts_missing"] = (
    alerts[required_alert_fields].isna().sum().sort_values(ascending=False)
    if not alerts.empty
    else pd.Series(dtype=int)
)

# Time gaps
GAP_THRESHOLD_MINUTES = 30

event_gaps = None
if events["ts_dt"].notna().any():
    events_sorted = events.sort_values("ts_dt")
    event_deltas = events_sorted["ts_dt"].diff().dt.total_seconds().div(60).fillna(0)
    event_gaps = event_deltas[event_deltas > GAP_THRESHOLD_MINUTES]

alert_gaps = None
if alerts["alert_ts_dt"].notna().any():
    alerts_sorted = alerts.sort_values("alert_ts_dt")
    alert_deltas = alerts_sorted["alert_ts_dt"].diff().dt.total_seconds().div(60).fillna(0)
    alert_gaps = alert_deltas[alert_deltas > GAP_THRESHOLD_MINUTES]

# Duplicates
event_dupes = events.duplicated(
    subset=["ts_epoch", "source", "event_type", "user", "host", "ip"], keep=False
).sum()
alert_dupes = alerts.duplicated(
    subset=["rule_id", "severity", "group", "evidence"], keep=False
).sum()

print("Missing fields (events):")
print(quality["events_missing"])
print("
Missing fields (alerts):")
print(quality["alerts_missing"])

print(f"
Event duplicates: {event_dupes:,}")
print(f"Alert duplicates: {alert_dupes:,}")

if event_gaps is not None:
    print(f"
Event gaps > {GAP_THRESHOLD_MINUTES} minutes: {len(event_gaps):,}")
    print(f"Max event gap (minutes): {event_gaps.max():.2f}" if not event_gaps.empty else "Max event gap: n/a")
else:
    print("
Event gaps: n/a (no timestamps)")

if alert_gaps is not None:
    print(f"Alert gaps > {GAP_THRESHOLD_MINUTES} minutes: {len(alert_gaps):,}")
    print(f"Max alert gap (minutes): {alert_gaps.max():.2f}" if not alert_gaps.empty else "Max alert gap: n/a")
else:
    print("Alert gaps: n/a (no timestamps)")


## Visualizations


In [ ]:
# Alerts over time
if alerts["alert_ts_dt"].notna().any():
    alerts_ts = alerts.dropna(subset=["alert_ts_dt"]).set_index("alert_ts_dt")
    series = alerts_ts.resample("1H").size()
    plt.figure(figsize=(10, 4))
    plt.plot(series.index, series.values, color="steelblue")
    plt.title("Alerts over time")
    plt.xlabel("Time")
    plt.ylabel("Alerts")
    plt.tight_layout()
    plt.show()
else:
    print("No alert timestamps available for time series plot.")


In [ ]:
# Top rule_id
if not alerts.empty:
    top_rules = alerts["rule_id"].value_counts().head(10)
    plt.figure(figsize=(8, 4))
    top_rules.plot(kind="bar", color="slateblue")
    plt.title("Top rule_id")
    plt.xlabel("rule_id")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No alerts available for rule_id plot.")


In [ ]:
# Top IPs, Users, Hosts
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

if "ip" in events.columns:
    events["ip"].value_counts().head(10).plot(kind="bar", ax=axes[0], color="teal")
    axes[0].set_title("Top IPs")
    axes[0].set_xlabel("IP")
    axes[0].set_ylabel("Count")

if "user" in events.columns:
    events["user"].value_counts().head(10).plot(kind="bar", ax=axes[1], color="orange")
    axes[1].set_title("Top Users")
    axes[1].set_xlabel("User")
    axes[1].set_ylabel("Count")

if "host" in events.columns:
    events["host"].value_counts().head(10).plot(kind="bar", ax=axes[2], color="purple")
    axes[2].set_title("Top Hosts")
    axes[2].set_xlabel("Host")
    axes[2].set_ylabel("Count")

plt.tight_layout()
plt.show()


In [ ]:
# GeoIP map (scatter using lat/lon)
geo_points = alerts.dropna(subset=["geo_lat", "geo_lon"])
if not geo_points.empty:
    plt.figure(figsize=(10, 5))
    plt.scatter(geo_points["geo_lon"], geo_points["geo_lat"], s=20, alpha=0.6, color="darkred")
    plt.title("GeoIP distribution (alerts)")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.tight_layout()
    plt.show()
else:
    print("No GeoIP lat/lon available for map visualization.")


## Feature Engineering


In [ ]:
# Normalize base fields
features = events.copy()
features["event_type"] = features["event_type"].fillna("unknown").astype(str)
features["source"] = features["source"].fillna("unknown").astype(str)
features["user"] = features["user"].fillna("unknown").astype(str)
features["host"] = features["host"].fillna("unknown").astype(str)
features["ip"] = features["ip"].fillna("unknown").astype(str)

features["ts_dt"] = pd.to_datetime(features["ts_dt"], errors="coerce")

features["user_5m_count"] = np.nan
features["user_1h_count"] = np.nan
features["host_5m_count"] = np.nan
features["host_1h_count"] = np.nan

valid = features[features["ts_dt"].notna()].copy()
valid = valid.sort_values("ts_dt")

if not valid.empty:
    valid["user_5m_count"] = (
        valid.groupby("user")
        .rolling("5min", on="ts_dt")["ts_dt"]
        .count()
        .reset_index(level=0, drop=True)
    )
    valid["user_1h_count"] = (
        valid.groupby("user")
        .rolling("1h", on="ts_dt")["ts_dt"]
        .count()
        .reset_index(level=0, drop=True)
    )
    valid["host_5m_count"] = (
        valid.groupby("host")
        .rolling("5min", on="ts_dt")["ts_dt"]
        .count()
        .reset_index(level=0, drop=True)
    )
    valid["host_1h_count"] = (
        valid.groupby("host")
        .rolling("1h", on="ts_dt")["ts_dt"]
        .count()
        .reset_index(level=0, drop=True)
    )

    for col in ["user_5m_count", "user_1h_count", "host_5m_count", "host_1h_count"]:
        features.loc[valid.index, col] = valid[col]

features["unique_ips_per_user"] = features.groupby("user")["ip"].transform("nunique")
features["unique_countries_per_user"] = features.groupby("user")["geo_country"].transform("nunique")

features[[
    "user_5m_count",
    "user_1h_count",
    "host_5m_count",
    "host_1h_count",
    "unique_ips_per_user",
    "unique_countries_per_user",
]] = features[[
    "user_5m_count",
    "user_1h_count",
    "host_5m_count",
    "host_1h_count",
    "unique_ips_per_user",
    "unique_countries_per_user",
]].fillna(0)

features.head()


In [ ]:
# Categorical encoding preview
categorical_cols = ["event_type", "source"]
encoded_preview = pd.get_dummies(features[categorical_cols], prefix=categorical_cols, dummy_na=False)
encoded_preview.head()


## ML Baseline: RandomForest


In [ ]:
label_candidates = [
    "label",
    "is_malicious",
    "malicious",
    "is_anomaly",
    "anomaly",
    "target",
    "y",
    "alerted",
]
label_col = None
for candidate in label_candidates:
    if candidate in features.columns:
        label_col = candidate
        break

alerts_ip_set = set(alerts["alert_ip"].dropna().astype(str))

if label_col is None:
    # Pseudo-labels using rules + burst anomalies
    rare_event_types = features["event_type"].value_counts(normalize=True)
    rare_event_types = set(rare_event_types[rare_event_types < 0.01].index)

    q_user_5m = features["user_5m_count"].quantile(0.99)
    q_host_5m = features["host_5m_count"].quantile(0.99)

    features["pseudo_label"] = (
        features["event_type"].isin(rare_event_types)
        | features["ip"].isin(alerts_ip_set)
        | (features["user_5m_count"] > q_user_5m)
        | (features["host_5m_count"] > q_host_5m)
    ).astype(int)
    label_col = "pseudo_label"
    print("No explicit labels found. Using pseudo-labels.")
else:
    print(f"Using label column: {label_col}")

model_features = [
    "user_5m_count",
    "user_1h_count",
    "host_5m_count",
    "host_1h_count",
    "unique_ips_per_user",
    "unique_countries_per_user",
    "event_type",
    "source",
]

feature_df = features[model_features + [label_col]].copy()
feature_df = feature_df.dropna(subset=[label_col])

X = feature_df[model_features]
y = feature_df[label_col].astype(int)

categorical_features = ["event_type", "source"]
numeric_features = [
    "user_5m_count",
    "user_1h_count",
    "host_5m_count",
    "host_1h_count",
    "unique_ips_per_user",
    "unique_countries_per_user",
]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features),
    ]
)

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
)

rf_pipeline = Pipeline(steps=[("preprocess", preprocess), ("model", rf_model)])

rf_metrics = {}
rf_pred_full = None

if y.nunique() < 2:
    print("Not enough label variety to train RandomForest.")
else:
    stratify = y if y.nunique() > 1 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=stratify
    )

    rf_pipeline.fit(X_train, y_train)
    y_pred = rf_pipeline.predict(X_test)
    y_proba = rf_pipeline.predict_proba(X_test)[:, 1]

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="binary", zero_division=0
    )

    try:
        roc_auc = roc_auc_score(y_test, y_proba)
    except ValueError:
        roc_auc = float("nan")

    rf_metrics = {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
    }

    print("RandomForest metrics:")
    print(rf_metrics)

    # Predictions on full dataset for comparison table
    rf_pred_full = rf_pipeline.predict(X)


## Advanced Model: GRU Autoencoder


In [ ]:
try:
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    from app.ml.gru_model import GRUAutoencoder
    TORCH_AVAILABLE = True
except Exception as exc:
    TORCH_AVAILABLE = False
    print(f"Torch not available, skipping GRU autoencoder: {exc}")

if TORCH_AVAILABLE:
    sequence_length = 20
    hidden_size = 32
    latent_size = 16
    epochs = 5
    batch_size = 64

    seq_df = features.dropna(subset=["ts_dt"]).copy()
    seq_df["entity_key"] = seq_df["user"].fillna("unknown") + "|" + seq_df["host"].fillna("unknown")

    event_type_map = {k: i + 1 for i, k in enumerate(sorted(seq_df["event_type"].unique()))}
    source_map = {k: i + 1 for i, k in enumerate(sorted(seq_df["source"].unique()))}

    sequences = []
    sequence_indices = []

    for entity_key, group in seq_df.groupby("entity_key"):
        group = group.sort_values("ts_dt")
        vectors = []
        indices = []
        for idx, row in group.iterrows():
            hour = row["ts_dt"].hour if pd.notna(row["ts_dt"]) else 0
            vec = np.array(
                [
                    event_type_map.get(row["event_type"], 0),
                    source_map.get(row["source"], 0),
                    hour,
                    row["user_5m_count"],
                    row["host_5m_count"],
                    row["unique_ips_per_user"],
                    row["unique_countries_per_user"],
                ],
                dtype=np.float32,
            )
            vectors.append(vec)
            indices.append(idx)

        if not vectors:
            continue

        for start in range(0, len(vectors), sequence_length):
            chunk = vectors[start : start + sequence_length]
            idx_chunk = indices[start : start + sequence_length]
            if len(chunk) < sequence_length:
                pad_len = sequence_length - len(chunk)
                chunk.extend([np.zeros_like(chunk[0]) for _ in range(pad_len)])
                idx_chunk.extend([None] * pad_len)
            sequences.append(np.stack(chunk))
            sequence_indices.append(idx_chunk)

    if len(sequences) < 5:
        print("Not enough sequences to train GRU autoencoder.")
        gru_scores = None
        gru_threshold = None
        gru_flags = None
    else:
        sequences = np.stack(sequences)

        scaler = StandardScaler()
        flat = sequences.reshape(-1, sequences.shape[-1])
        scaled = scaler.fit_transform(flat).reshape(sequences.shape)

        dataset = TensorDataset(torch.tensor(scaled, dtype=torch.float32))
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        model = GRUAutoencoder(input_size=scaled.shape[-1], hidden_size=hidden_size, latent_size=latent_size)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        loss_fn = torch.nn.MSELoss()

        model.train()
        for epoch in range(epochs):
            losses = []
            for (batch,) in loader:
                optimizer.zero_grad()
                recon = model(batch)
                loss = loss_fn(recon, batch)
                loss.backward()
                optimizer.step()
                losses.append(loss.item())
            print(f"Epoch {epoch + 1}/{epochs} loss={np.mean(losses):.4f}")

        model.eval()
        with torch.no_grad():
            recon = model(torch.tensor(scaled, dtype=torch.float32))
            mse = torch.mean((recon - torch.tensor(scaled, dtype=torch.float32)) ** 2, dim=(1, 2))
            gru_scores = mse.numpy()

        gru_threshold = float(np.quantile(gru_scores, 0.99))
        print(f"GRU anomaly threshold (0.99 quantile): {gru_threshold:.6f}")

        # Flag events associated with anomalous sequences
        gru_flags = pd.Series(0, index=features.index)
        for score, idx_chunk in zip(gru_scores, sequence_indices):
            if score > gru_threshold:
                for idx in idx_chunk:
                    if idx is not None:
                        gru_flags.loc[idx] = 1


## Comparison Table


In [ ]:
comparison_rows = []

labels = None
if label_col in feature_df.columns:
    labels = feature_df[label_col].astype(int)


def compute_rates(pred, labels_series):
    pred = pd.Series(pred, index=labels_series.index).astype(int)
    labels_series = labels_series.astype(int)
    tp = ((pred == 1) & (labels_series == 1)).sum()
    fp = ((pred == 1) & (labels_series == 0)).sum()
    tn = ((pred == 0) & (labels_series == 0)).sum()
    fn = ((pred == 0) & (labels_series == 1)).sum()
    detection = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    fp_rate = fp / (fp + tn) if (fp + tn) > 0 else float("nan")
    return detection, fp_rate

if labels is not None and labels.nunique() > 1:
    rule_flags = features.loc[labels.index, "ip"].isin(alerts_ip_set).astype(int)
    detection, fp_rate = compute_rates(rule_flags, labels)
    comparison_rows.append(
        {
            "method": "rules",
            "detection_rate_proxy": detection,
            "fp_proxy": fp_rate,
        }
    )

    if rf_pred_full is not None:
        detection, fp_rate = compute_rates(rf_pred_full, labels)
        comparison_rows.append(
            {
                "method": "random_forest",
                "detection_rate_proxy": detection,
                "fp_proxy": fp_rate,
            }
        )

    if TORCH_AVAILABLE and "gru_flags" in globals() and gru_flags is not None:
        detection, fp_rate = compute_rates(gru_flags.loc[labels.index], labels)
        comparison_rows.append(
            {
                "method": "gru_autoencoder",
                "detection_rate_proxy": detection,
                "fp_proxy": fp_rate,
            }
        )
else:
    print("Not enough label variety to compute comparison metrics.")

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


## Recommendations


In [ ]:
recs = []

recs.append("- Tune alert thresholds using percentile baselines (for example, 0.99 quantile on GRU scores or burst counts).")
recs.append("- Add correlation rules for top IPs, users, and hosts that repeatedly surface in alerts.")
recs.append("- Enrich and normalize GeoIP so country and ASN coverage is consistent across events and alerts.")
recs.append("- Reduce data gaps by monitoring ingestion latency and backfilling missing windows.")
recs.append("- Standardize event_type/source naming to reduce sparsity in categorical features.")

from IPython.display import Markdown, display

display(Markdown("
".join(recs)))
